In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import SparsePauliOp

from estimator import MyEstimator

from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

from itertools import product

import hamiltonian_generator, evolution_circuits

import warnings
warnings.filterwarnings("ignore")

In [2]:
service = QiskitRuntimeService()
backend_name = 'ibm_kingston'
backend = service.backend(backend_name)

qiskit_runtime_service.__init__:WARNING:2026-05-04 18:32:34,758: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (premium), the available account instances are: m5000-eu, m5000-us. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-05-04 18:32:34,759: Using instance: m5000-us, plan: premium


In [3]:
sampler = Sampler(backend)

In [4]:
reps = [0, 1, 2]
num_sites = [5, 12]
couplings = [0.5, 5]
dd_enabled = [False, True]
folds = [1, 3, 5]

final_time = 0.25
num_timesteps = 5

In [5]:
for (rep, n, h, dd, f) in product(reps, num_sites, couplings, dd_enabled, folds) :
    dd_str = ('dd' if dd else 'std')
    job_name = f'rep_{rep}_evo_job_{n}_{h}_{dd_str}_{f}'

    circuit = evolution_circuits.dynamic_evolve(
        n, h, final_time, num_timesteps, mcm=True, stretch_dd=dd, folds=f)

    dynamic_hamiltonian = hamiltonian_generator.get_dynamic_heisenberg_hamiltonian(n, h)

    estimator = MyEstimator(
        job_name, circuit, None, dynamic_hamiltonian, sampler, service, backend)

    print(f'Submitting {job_name}')
    estimator.submit_sampler_jobs()

Submitting rep_0_evo_job_5_0.5_std_1
Sampler jobs already submitted. Result in jobs/rep_0_evo_job_5_0.5_std_1.json.
Submitting rep_0_evo_job_5_0.5_std_3
Sampler jobs already submitted. Result in jobs/rep_0_evo_job_5_0.5_std_3.json.
Submitting rep_0_evo_job_5_0.5_std_5
Sampler jobs already submitted. Result in jobs/rep_0_evo_job_5_0.5_std_5.json.
Submitting rep_0_evo_job_5_0.5_dd_1
Sampler jobs already submitted. Result in jobs/rep_0_evo_job_5_0.5_dd_1.json.
Submitting rep_0_evo_job_5_0.5_dd_3
Sampler jobs already submitted. Result in jobs/rep_0_evo_job_5_0.5_dd_3.json.
Submitting rep_0_evo_job_5_0.5_dd_5
Sampler jobs already submitted. Result in jobs/rep_0_evo_job_5_0.5_dd_5.json.
Submitting rep_0_evo_job_5_5_std_1
Sampler jobs already submitted. Result in jobs/rep_0_evo_job_5_5_std_1.json.
Submitting rep_0_evo_job_5_5_std_3
Sampler jobs already submitted. Result in jobs/rep_0_evo_job_5_5_std_3.json.
Submitting rep_0_evo_job_5_5_std_5
Sampler jobs already submitted. Result in jobs/rep_

In [6]:
for (rep, n, h, dd, f) in product(reps, num_sites, couplings, dd_enabled, folds) :
    dd_str = ('dd' if dd else 'std')
    job_name = f'rep_{rep}_evo_job_{n}_{h}_{dd_str}_{f}'

    circuit = evolution_circuits.dynamic_evolve(
        n, h, final_time, num_timesteps, mcm=True, stretch_dd=dd, folds=f)

    dynamic_hamiltonian = hamiltonian_generator.get_dynamic_heisenberg_hamiltonian(n, h)

    estimator = MyEstimator(
        job_name, circuit, None, dynamic_hamiltonian, sampler, service, backend)

    print(f'Evaluating {job_name}')
    energy = estimator.evaluate_expectation_value()
    print(f"{job_name}: \t\t {energy}")
    np.save(f'data/rep_{rep}_evo_energy_{n}_{h}_{dd_str}_{f}', energy)

Evaluating rep_0_evo_job_5_0.5_std_1
rep_0_evo_job_5_0.5_std_1: 		 0.613525390625
Evaluating rep_0_evo_job_5_0.5_std_3
rep_0_evo_job_5_0.5_std_3: 		 0.1318359375
Evaluating rep_0_evo_job_5_0.5_std_5
rep_0_evo_job_5_0.5_std_5: 		 0.4736328125
Evaluating rep_0_evo_job_5_0.5_dd_1
rep_0_evo_job_5_0.5_dd_1: 		 0.70361328125
Evaluating rep_0_evo_job_5_0.5_dd_3
rep_0_evo_job_5_0.5_dd_3: 		 0.50146484375
Evaluating rep_0_evo_job_5_0.5_dd_5
rep_0_evo_job_5_0.5_dd_5: 		 0.32568359375
Evaluating rep_0_evo_job_5_5_std_1
rep_0_evo_job_5_5_std_1: 		 -3.044921875
Evaluating rep_0_evo_job_5_5_std_3
rep_0_evo_job_5_5_std_3: 		 1.35302734375
Evaluating rep_0_evo_job_5_5_std_5
rep_0_evo_job_5_5_std_5: 		 2.48828125
Evaluating rep_0_evo_job_5_5_dd_1
rep_0_evo_job_5_5_dd_1: 		 0.64111328125
Evaluating rep_0_evo_job_5_5_dd_3
rep_0_evo_job_5_5_dd_3: 		 1.705078125
Evaluating rep_0_evo_job_5_5_dd_5
rep_0_evo_job_5_5_dd_5: 		 1.63037109375
Evaluating rep_0_evo_job_12_0.5_std_1
rep_0_evo_job_12_0.5_std_1: 		 3.